# How to generate a metareport?

### Create a metareport comparing synthetic datasets with respect to a list of metrics. /!\ Only for the summary.

Assume that the synthetic datasets to compare are already generated \
Based on the Wisconsin Breast Cancer Dataset

In [1]:
# Standard library
import sys
import tempfile
from pathlib import Path

sys.path.append("..")

# 3rd party packages
import pandas as pd

# Local packages
import config
import utils.draw
from metrics.metareport import Metareport

## Load the real and synthetic WBCD datasets

In [2]:
df_real = {}
df_real["train"] = pd.read_csv(
    "../data/" + config.WBCD_DATASET_TRAIN_FILEPATH.stem + ".csv"
)
df_real["test"] = pd.read_csv(
    "../data/" + config.WBCD_DATASET_TEST_FILEPATH.stem + ".csv"
)
df_real["train"].shape

(359, 10)

### Choose the synthetic dataset - No need for splitting train/test here, will be done by the metareport

In [3]:
df_synth = pd.concat(  # generated by Synthpop here
    [
        pd.read_csv("../results/data/2023-07-31_Synthpop_359samples.csv"),
        pd.read_csv("../results/data/2023-07-31_Synthpop_90samples.csv"),
    ],
    axis=0,
    ignore_index=True,
)

df_mock = (
    pd.concat(  # random synthetic dataset to compare to the one generated by Synthpop
        [
            df_real["train"].apply(lambda x: x.sample(frac=1, replace=True).to_numpy()),
            df_real["test"].apply(lambda x: x.sample(frac=1, replace=True).to_numpy()),
        ],
        axis=0,
        ignore_index=True,
    )
)

## Configure the metadata dictionary

### The continuous and categorical variables need to be specified, as well as the variable to predict for the future learning task

In [4]:
metadata = {
    "continuous": [
        "Clump_Thickness",
        "Uniformity_of_Cell_Size",
        "Uniformity_of_Cell_Shape",
        "Marginal_Adhesion",
        "Single_Epithelial_Cell_Size",
        "Bland_Chromatin",
        "Normal_Nucleoli",
        "Mitoses",
        "Bare_Nuclei",
    ],
    "categorical": ["Class"],
    "variable_to_predict": "Class",
}

## Generate the metareport

Some metrics will not be computed since the only categorical variable is the variable to predict 

In [5]:
parameters = {  # see the notebooks utility_report and privacy_report for more details
    "cross_learning": True,
    "num_repeat": 3,
    "use_gpu": True,
    "sampling_frac": 0.2,
}

In [6]:
with tempfile.TemporaryDirectory() as temp_dir:
    df_synth.to_csv(Path(temp_dir) / "synthpop.csv", index=False)
    df_mock.to_csv(Path(temp_dir) / "random.csv", index=False)

    metareport = Metareport(
        dataset_name="Wisconsin Breast Cancer Dataset",
        df_real=df_real,
        synthetic_data_path=temp_dir,  # the datasets to compare need to be saved in the same folder
        metadata=metadata,
        figsize=(8, 6),  # will be automatically adjusted for larger or longer figures
        random_state=0,  # for reproducibility purposes
        metareport_folderpath=None,  # a dictionary containing the path of each already computed report to load and compare
        metrics=None,  # list of the metrics to compute. Can be utility or privacy metrics. If not specified, all the metrics are computed.
        params=parameters,  # the dictionary containing the parameters for both utility and privacy reports
    )

In [7]:
metareport.compute()

## Get the summary report as a pandas dataframe

In [8]:
df_summary = metareport.summary()

### Style the result

The best value (minimum or maximal according to the submetric objective) is colored in green. The worst in yellow.

In [9]:
s = df_summary.style.pipe(Metareport.make_pretty, metrics=list(df_summary.index))
s

compared,random,synthpop
metric,,
cat_consis-within_ratio,1.00,1.00
cat_stats-frequency_coverage,0.98,0.97
cat_stats-support_coverage,1.00,1.00
classif-diff_real_synth,0.73,0.02
cont_consis-within_ratio,1.00,1.00
cont_stats-iqr_l1_distance,0.02,0.06
cont_stats-median_l1_distance,0.01,0.01
cross_classif-diff_real_synth,0.73,0.02
cross_reg-diff_real_synth,8.74,0.67


### Save the styled result as html

In [10]:
with tempfile.TemporaryDirectory() as temp_dir:
    with open(Path(temp_dir) / "df.html", "w") as f:
        print(s.to_html(), file=f)

## Save and load the metareport

In [11]:
with tempfile.TemporaryDirectory() as temp_dir:
    metareport.save(savepath=temp_dir)  # save
    new_report = Metareport(
        metareport_folderpath={"synthpop": temp_dir, "random": temp_dir}
    )  # load